# GraphOPF code

In [ ]:
# !pip install torch_geometric
# !pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-2.5.0+cu121.html

# !pip install pypower
# !pip install pyrlu
# # !pip install conflictfree

In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

In [1]:
import os
import torch
import numpy as np
import random
import time

# def set_seed(seed):
#     os.environ["PYTHONHASHSEED"] = str(seed)
#     random.seed(seed)
#     np.random.seed(seed)
#     torch.manual_seed(seed)
#     torch.cuda.manual_seed(seed)
#     torch.cuda.manual_seed_all(seed)

#     # 완전한 재현성을 위해
#     torch.backends.cudnn.deterministic = True
#     torch.backends.cudnn.benchmark = False
#     # torch.use_deterministic_algorithms(True)

# # 3 random seeds
# seed_list = [0, 1, 2]
# set_seed(seed_list[2])

os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"]= "0"
# os.environ["CUDA_VISIBLE_DEVICES"] = '0, 1, 2, 3'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('Device:', device)  # 출력결과: cuda
print('Count of using GPUs:', torch.cuda.device_count())   #출력결과: 1 (GPU #2 한개 사용하므로)
print('Current cuda device:', torch.cuda.current_device())  # 출력결과: 2 (GPU #2 의미)


Device: cuda
Count of using GPUs: 1
Current cuda device: 0


In [3]:
# %cd /content/drive/MyDrive/kj/GOC3970_case_real_slack
# !pwd

In [2]:
import pickle

from utils.utils3970_graphlde import ACOPFProblem
# from utils.utils3970_graphlde_test import ACOPFProblem

filepath = './data/FeasiblePairs_Case3970_20_perturb_10000_samples.mat'

data = ACOPFProblem(filename=filepath) # call ACOPFProblem class in the utils.py <== In DeepLDE code, need to modify! so messy...

save_data = False

# check the size of train/validation/test dataset.
# print("Dataset of GraphLDE: ")
# print(data.train_dataset)
# print(problem.valid_dataset)
# print(problem.test_dataset)

DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
data._device = DEVICE
# Put all variables in "data" to the cuda.
for attr in dir(data):
    var = getattr(data, attr)
    if not callable(var) and not attr.startswith("__") and torch.is_tensor(var):
        try:
            setattr(data, attr, var.to(DEVICE))
        except AttributeError:
            pass


/global/u1/k/kjsong/FedOPF-APPFL/fine-tuning-task/unseen/Case3970/utils/utils3970_graphlde.py:109: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  self.slackva = torch.tensor([np.deg2rad(ppc['bus'][self.slack, idx_bus.VA])],


In [14]:
import torch
import torch_geometric
torch.cuda.empty_cache()
import torch.optim as optim
torch.set_default_dtype(torch.float32) #  If the inputs are torch.float32, must be torch.complex64. If the inputs are torch.float64, must be torch.complex128.

from torch.utils.data import TensorDataset, DataLoader, Dataset

import pickle

from pypower.api import loadcase

from model.Edge_GNN_solver import Edge_GNNSolver

from utils.loss_fn_graphlde import total_loss, ineq_violation
from utils.log import dict_agg
from global_config import base_config, global_logger, ROOT_DIRECTORY, logging
from pathlib import Path
import pickle

# from conflictfree.grad_operator import ConFIG_update
# from conflictfree.momentum_operator import PseudoMomentumOperator
# # from conflictfree.grad_operator import ConFIGOperator
# from conflictfree.utils import get_gradient_vector,apply_gradient_vector

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# # Set the linear algebra solver(?)
# torch._C._set_linalg_preferred_backend(torch._C._LinalgBackend.Magma) # torch._C._LinalgBackend.Magma, torch._C._LinalgBackend.Cusolver
# torch._C._get_linalg_preferred_backend()

# MAGMA_NOWARNING
# torch.set_warn_always(False)


* First, let's see the information of ACOPF for targeted power network!

In [15]:
ppc = loadcase("./data/matpower/pglib_opf_case3970_goc.mat") # in this code, we used Pypower for loading benchmark power network.
## NOTE: the dataset we used are Pypower and PGLib, so we additionally need to check whether the targeted power network is same or not!
## e.g., IEEE 57case in Pypower has different "rate A", "rate B", "rate C" values compared to PGLib.

ng = ppc['gen'].shape[0] # number of generators.
nbus = ppc['bus'].shape[0] # number of buses.
nl = ppc['branch'].shape[0] # total number of branches and transformers.

In [16]:
from scipy.stats.qmc import LatinHypercube

train_config = {
    'probType': 'acopf',
    'useCompl': True, # boolean type: whether to use completion (DC3, DeepLDE 기술)

    # GNN model parameters
    'n_gnn_layers': 3,
    'nfeature_dim': 2, # input dim
    'efeature_dim': 4, # edge feature dim 
    'hidden_dim': 40,
    'dropout_rate': 0.1,
    'K': 10, # 6 # only for TAGConv or ChebConv and GATConv (as multi-head)

    # DeepLDE hyperparameters
    'epochs': 30, # 10 (GPU RTX 4090), 8 (GPU A100)
    'batchSize': 5, # 6, (8) (GPU RTX 4090; GPU 다운 에러 발생..), 16 (GPU A100)
    'lr': 95e-5, # 1e-4
    'lr_w': 95e-5, # 1e-4
    'weight_decay': 1e-5,

    # LDF parameters
    'rho_init': 0.0009, # 0.0008, 
    's_init': 0.1,
    'p_iter_max': 10,
    'warmup_iter': 0, # 20, # 0 for non-warmup start cases
    'corrEps': 1e-4, # float type: correction procedure tolerance
}


eps_converge = train_config['corrEps']
valid_eps_converge = 1e-4
nepochs = train_config['epochs']
batch_size = train_config['batchSize']

train_loss_list = []
train_ineq_max = []
train_ineq_mean = []
valid_loss_list = []
valid_eval_list = []

Kshot = 20 # 20
train_len_range = (0,Kshot) ## K-shot: 20, 10, 5, 1, 0
node_means, node_stds, edge_means, edge_stds = data.input_standardization(train_len_range) # (1, 2*nbus) <= for data normalization
n_means = node_means.to(DEVICE)
n_stds = node_stds.to(DEVICE)
e_means = edge_means.to(DEVICE)
e_stds = edge_stds.to(DEVICE)

## Random sampling
# train_loader = torch_geometric.loader.DataLoader(random.sample(data.train_dataset, 100), batch_size=train_config['batchSize'], shuffle=True, drop_last=True)

## Slicing
train_loader = torch_geometric.loader.DataLoader(data.train_dataset[train_len_range[0]:train_len_range[1]], batch_size=train_config['batchSize'], shuffle=True, drop_last=True)

valid_loader = torch_geometric.loader.DataLoader(data.valid_dataset, batch_size=train_config['batchSize'], shuffle=False, drop_last=True)

# solver_net = GNNSolver(data, train_config)
solver_net = Edge_GNNSolver(data, train_config)

solver_net.to(DEVICE)

print(solver_net)

Edge_GNNSolver(
  (layers): ModuleList(
    (0): EdgeAggregation()
    (1): TransformerConv(120, 40, heads=10)
    (2): EdgeAggregation()
    (3): TransformerConv(120, 40, heads=10)
    (4): EdgeAggregation()
    (5): TransformerConv(120, 40, heads=10)
  )
  (flatten): Linear(in_features=14760, out_features=245, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
)


In [17]:
#################################### SETTING THE LOGS ####################################
result_path = os.path.join(ROOT_DIRECTORY, "results")
# folder_name = "FT_unseen_data_" + str(Kshot) + "_shot_" + str(train_config["batchSize"]) + "_bs_" + str(train_config["epochs"]) + "_epochs_" + str(train_config["lr"]) + "_lr_" \
#               + str(train_config["rho_init"]) + "_rho_init"
folder_name = "Local_unseen_data_" + str(Kshot) + "_shot_" + str(train_config["batchSize"]) + "_bs_" + str(train_config["epochs"]) + "_epochs_" + str(train_config["lr"]) + "_lr_" \
               + str(train_config["rho_init"]) + "_rho_init"

log_path = os.path.join(result_path, folder_name, "logs")
data_tracking_path = os.path.join(result_path, folder_name, "data_tracking")

Path(log_path).mkdir(parents=True, exist_ok=True)
Path(data_tracking_path).mkdir(parents=True, exist_ok=True)
fileh = logging.FileHandler(os.path.join(log_path, "log.txt"), 'a')
global_logger.addHandler(fileh)

* Load pretrained GraphOPF model

In [7]:
pretrained_global_graphlde = torch.load("./model/global_model/checkpoint_Global.pth", weights_only=False)

* Transfer the weights from the pretrained GraphOPF

In [8]:
global_pretrained_state_dict = pretrained_global_graphlde
target_state_dict = solver_net.state_dict()

layer_names = list(target_state_dict.keys())
# load global model
for name in layer_names[:-2]:
    if name in target_state_dict.keys() and name in global_pretrained_state_dict.keys():
        # print(name)
        target_state_dict[name] = global_pretrained_state_dict[name].clone()

solver_net.load_state_dict(target_state_dict)

# # Freezing the half-GNN layers
# for i, (name, param) in enumerate(solver_net.named_parameters()):
#     if i <= 32: # 5, 10, 21
#         # print(name)
#         param.requires_grad = False
#     else:
#         print(name)
#         param.requires_grad = True

<All keys matched successfully>

* Training method: LD framework

In [18]:
stats = {}

# NOTE: LDF parameters.
LagM_sp_gen = torch.ones(1, 2).to(DEVICE) # shape: (1, num_inequalities)
LagM_gen = torch.ones(1, 2*ng).to(DEVICE) # shape: (1, num_inequalities)
LagM_bus = torch.ones(1, 2*nbus).to(DEVICE) # shape: (1, num_inequalities)
LagM_line = torch.ones(1, 2*nl).to(DEVICE) # shape: (1, num_inequalities)

warmup_iter = train_config["warmup_iter"] # the warmup period: the NN is trained with an additional inner iteration before the first outer iteration.
rho_init = train_config["rho_init"]
s_init = train_config["s_init"]

rho = rho_init
s = s_init

rho_iter = 0
s_iter = 0

p_iter_max = train_config["p_iter_max"]
p_iter_max_sum = p_iter_max

d = 0 # 0 for static case otherwise use 5"
beta = 0 # 0.001 # 0 for static case otherwise use 1

lr_w = train_config["lr_w"] # 이거 증가해도 되지 않을지?
lr = train_config["lr"]

print_interval = 1
epoch_stats = {}
for i in range(nepochs):
    ################### TRAINING PHASE ###################
    solver_net.train()
    if i<warmup_iter:
        ######### WARM-UP PERIOD #########
        if i == 0:
            solver_opt = optim.Adam(solver_net.parameters(), lr=lr_w, weight_decay=train_config["weight_decay"]) # this will be reinitalized after warmup stage
            print("Warmup start!")

        for Xtrain in train_loader:
            Xtrain = Xtrain.to(DEVICE)
            # start_time = time.time()
            solver_opt.zero_grad()

            # DNN + NR (equality constraint)
            Yhat_train = solver_net(Xtrain, n_means, n_stds, e_means, e_stds) # solve power flow equation using Newton-Rahpson method (NOTE: if "useCompl" is True.)
            # Yhat_train = solver_net(Xtrain, n_means, n_stds) # solve power flow equation using Newton-Rahpson method (NOTE: if "useCompl" is True.)

            # train_loss, train_obj, ineq_dist, eq_resid = total_loss(data, Xtrain.x, Yhat_train, LagM) # LagM is lagrangian multiplier, and the shape is (1, num_inequalities)
            train_loss, train_obj, ineq_dist, eq_resid = total_loss(data, Xtrain.x, Yhat_train, LagM_sp_gen, LagM_gen, LagM_bus, LagM_line)

            train_loss.sum().backward()
            solver_opt.step()

            dict_agg(epoch_stats, 'train_loss', train_loss.detach().cpu().numpy())
            dict_agg(epoch_stats, 'train_obj', train_obj.detach().cpu().numpy())
            dict_agg(epoch_stats, 'train_ineq_max', torch.max(ineq_dist, dim=1)[0].detach().cpu().numpy())
            dict_agg(epoch_stats, 'train_ineq_mean', torch.mean(ineq_dist, dim=1).detach().cpu().numpy())

            ineq_p_g = ineq_dist[:,:2]
            ineq_q_g = ineq_dist[:,2:2+2*ng]
            ineq_v_m = ineq_dist[:,2+2*ng:2+2*ng+2*nbus]
            ineq_line_l = ineq_dist[:,2+2*ng+2*nbus:]

            dict_agg(epoch_stats, 'train_ineq_p_g_num_viol_0', torch.sum(ineq_p_g > eps_converge, dim=1).detach().cpu().numpy())
            dict_agg(epoch_stats, 'train_ineq_q_g_num_viol_0', torch.sum(ineq_q_g > eps_converge, dim=1).detach().cpu().numpy())
            dict_agg(epoch_stats, 'train_ineq_v_m_num_viol_0', torch.sum(ineq_v_m > eps_converge, dim=1).detach().cpu().numpy())
            dict_agg(epoch_stats, 'train_ineq_line_thermal_num_viol_0', torch.sum(ineq_line_l > eps_converge, dim=1).detach().cpu().numpy())

            dict_agg(epoch_stats, 'train_eq_max', torch.max(torch.abs(eq_resid), dim=1)[0].detach().cpu().numpy())
            dict_agg(epoch_stats, 'train_eq_mean', torch.mean(torch.abs(eq_resid), dim=1).detach().cpu().numpy())

    else:
        if i == warmup_iter:
            print("Warmup ended!")
            # data.ref_freedom = False

            # ineq_lag_flag = True # for the first warmup end epoch, consider lag. multipliers update of ineq.
        # elif (i - warmup_iter)%ineq_lag_flag_trigger == 0:
        #     # print("Doing test... continue this case..")
        #     print("consider lagrangian multipliers update for ineq. constraints!")
        #     ineq_lag_flag = True
        # else:
        #     ineq_lag_flag = False

        # for every (updated) p_iter_max_sum time.
        if (i - warmup_iter)%p_iter_max_sum == 0:
            ######### Outer Interation: calculate step size of lagrangian multipliers update #########
            if i> warmup_iter:
                print("current epoch %d || p_iter_max updated : %d -> %d" %(i, p_iter_max, p_iter_max + d))
                # s = s_init * (1/(1+beta*(s_iter + 1))) # 근데 이 부분 중복아닌가?? 있어야 하나????
                # s_iter += 1
                # print("mu iter updated : %d -> %d" %(s_iter-1, s_iter))

                rho = rho_init * (1/(1+beta*(rho_iter + 1))) # 근데 이 부분 중복아닌가?? 있어야 하나????
                rho_iter += 1
                print("rho iter updated : %d -> %d" %(rho_iter-1, rho_iter))
                p_iter_max = p_iter_max + d
                p_iter_max_sum += p_iter_max

            with torch.no_grad():
                print("Lambda updated at %d epoch" %i)
                solver_net.eval()
                for Xtrain in train_loader:
                    Xtrain = Xtrain.to(DEVICE)
                    #solver_opt.zero_grad()
                    Yhat_train = solver_net(Xtrain, n_means, n_stds, e_means, e_stds) # solve power flow equation using Newton-Rahpson method (NOTE: if "useCompl" is True.)
                    # Yhat_train = solver_net(Xtrain, n_means, n_stds) # solve power flow equation using Newton-Rahpson method (NOTE: if "useCompl" is True.)

                    # LagM += rho*ineq_violation(data, Xtrain.x, Yhat_train) # Update the lagrangian multiplier.
                    LagM_sp_gen += rho*ineq_violation(data, Xtrain.x, Yhat_train)[:2] # Update the lagrangian multiplier.
                    LagM_gen += rho*ineq_violation(data, Xtrain.x, Yhat_train)[2:2+2*ng] # Update the lagrangian multiplier.
                    LagM_bus += rho*ineq_violation(data, Xtrain.x, Yhat_train)[2+2*ng:2+2*ng+2*nbus] # Update the lagrangian multiplier.
                    LagM_line += rho*ineq_violation(data, Xtrain.x, Yhat_train)[2+2*ng+2*nbus:] # Update the lagrangian multiplier.

            solver_opt = optim.Adam(solver_net.parameters(), lr = lr, weight_decay=train_config["weight_decay"])

        solver_net.train()
        for Xtrain in train_loader:
            Xtrain = Xtrain.to(DEVICE)
            solver_opt.zero_grad()
            Yhat_train = solver_net(Xtrain, n_means, n_stds, e_means, e_stds) # solve power flow equation using Newton-Rahpson method (NOTE: if "useCompl" is True.)
            # Yhat_train = solver_net(Xtrain, n_means, n_stds) # solve power flow equation using Newton-Rahpson method (NOTE: if "useCompl" is True.)

            # train_loss, obj_train, ineq_dist, eq_resid = total_loss(data, Xtrain.x, Yhat_train, LagM) # LagM is lagrangian multiplier (1, num_inequalities)
            train_loss, train_obj, ineq_dist, eq_resid = total_loss(data, Xtrain.x, Yhat_train, LagM_sp_gen, LagM_gen, LagM_bus, LagM_line)

            train_loss.sum().backward()
            solver_opt.step()

            dict_agg(epoch_stats, 'train_loss', train_loss.detach().cpu().numpy())
            dict_agg(epoch_stats, 'train_obj', train_obj.detach().cpu().numpy())
            dict_agg(epoch_stats, 'train_ineq_max', torch.max(ineq_dist, dim=1)[0].detach().cpu().numpy())
            dict_agg(epoch_stats, 'train_ineq_mean', torch.mean(ineq_dist, dim=1).detach().cpu().numpy())

            ineq_p_g = ineq_dist[:,:2]
            ineq_q_g = ineq_dist[:,2:2+2*ng]
            ineq_v_m = ineq_dist[:,2+2*ng:2+2*ng+2*nbus]
            ineq_line_l = ineq_dist[:,2+2*ng+2*nbus:]

            dict_agg(epoch_stats, 'train_ineq_p_g_num_viol_0', torch.sum(ineq_p_g > eps_converge, dim=1).detach().cpu().numpy())
            dict_agg(epoch_stats, 'train_ineq_q_g_num_viol_0', torch.sum(ineq_q_g > eps_converge, dim=1).detach().cpu().numpy())
            dict_agg(epoch_stats, 'train_ineq_v_m_num_viol_0', torch.sum(ineq_v_m > eps_converge, dim=1).detach().cpu().numpy())
            dict_agg(epoch_stats, 'train_ineq_line_thermal_num_viol_0', torch.sum(ineq_line_l > eps_converge, dim=1).detach().cpu().numpy())

            dict_agg(epoch_stats, 'train_eq_max', torch.max(torch.abs(eq_resid), dim=1)[0].detach().cpu().numpy())
            dict_agg(epoch_stats, 'train_eq_mean', torch.mean(torch.abs(eq_resid), dim=1).detach().cpu().numpy())

    if (i == 0) or (i%print_interval == 0):
        print(
            'Epoch {}: train loss {:.4f}, train obj {:.4f}, ineq max {:.4f}, ineq mean {:.4f}, ineq p_g num viol {:.4f}, ineq q_g num viol {:.4f}, ineq v_m num viol {:.4f}, ineq line_theraml num viol {:.4f}, eq max {:.4f}, eq mean {:.4f}'.format(
                i, np.mean(epoch_stats['train_loss']), np.mean(epoch_stats['train_obj']), np.mean(epoch_stats['train_ineq_max']), np.mean(epoch_stats['train_ineq_mean']),
                np.mean(epoch_stats['train_ineq_p_g_num_viol_0']), np.mean(epoch_stats['train_ineq_q_g_num_viol_0']), np.mean(epoch_stats['train_ineq_v_m_num_viol_0']), np.mean(epoch_stats['train_ineq_line_thermal_num_viol_0']),
                np.mean(epoch_stats['train_eq_max']), np.mean(epoch_stats['train_eq_mean'])))

    global_logger.info('Epoch {}: train loss {:.4f}, train obj {:.4f}, ineq max {:.4f}, ineq mean {:.4f}, ineq p_g num viol {:.4f}, ineq q_g num viol {:.4f}, ineq v_m num viol {:.4f}, ineq line_theraml num viol {:.4f}, eq max {:.4f}, eq mean {:.4f}'.format(
                i, np.mean(epoch_stats['train_loss']), np.mean(epoch_stats['train_obj']), np.mean(epoch_stats['train_ineq_max']), np.mean(epoch_stats['train_ineq_mean']),
                np.mean(epoch_stats['train_ineq_p_g_num_viol_0']), np.mean(epoch_stats['train_ineq_q_g_num_viol_0']), np.mean(epoch_stats['train_ineq_v_m_num_viol_0']), np.mean(epoch_stats['train_ineq_line_thermal_num_viol_0']),
                np.mean(epoch_stats['train_eq_max']), np.mean(epoch_stats['train_eq_mean'])))

    train_loss_list.append(np.mean(epoch_stats['train_loss']))
    train_ineq_max.append(np.mean(epoch_stats['train_ineq_max']))
    train_ineq_mean.append(np.mean(epoch_stats['train_ineq_mean']))
    # valid_eval_list.append(np.mean(epoch_stats['valid_eval']))

Warmup ended!
Lambda updated at 0 epoch
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are desi

Epoch 0: train loss 95748.9453, train obj 3100.3618, ineq max 882.9482, ineq mean 0.4407, ineq p_g num viol 1.0000, ineq q_g num viol 41.3500, ineq v_m num viol 0.0000, ineq line_theraml num viol 327.7000, eq max 0.0010, eq mean 0.0000


Epoch 0: train loss 95748.9453, train obj 3100.3618, ineq max 882.9482, ineq mean 0.4407, ineq p_g num viol 1.0000, ineq q_g num viol 41.3500, ineq v_m num viol 0.0000, ineq line_theraml num viol 327.7000, eq max 0.0010, eq mean 0.0000
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good perform

Epoch 1: train loss 66563.5000, train obj 2360.9243, ineq max 632.4413, ineq mean 0.3230, ineq p_g num viol 1.0000, ineq q_g num viol 40.1500, ineq v_m num viol 0.0000, ineq line_theraml num viol 230.7250, eq max 0.0007, eq mean 0.0000


Epoch 1: train loss 66563.5000, train obj 2360.9243, ineq max 632.4413, ineq mean 0.3230, ineq p_g num viol 1.0000, ineq q_g num viol 40.1500, ineq v_m num viol 0.0000, ineq line_theraml num viol 230.7250, eq max 0.0007, eq mean 0.0000
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good perform

...

Epoch 27: train loss 6575.2432, train obj 731.9562, ineq max 203.9657, ineq mean 0.0581, ineq p_g num viol 0.9500, ineq q_g num viol 38.5054, ineq v_m num viol 0.0000, ineq line_theraml num viol 34.9589, eq max 0.0005, eq mean 0.0000
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performan

Epoch 28: train loss 6365.6602, train obj 722.7114, ineq max 197.1271, ineq mean 0.0561, ineq p_g num viol 0.9207, ineq q_g num viol 37.4948, ineq v_m num viol 0.0000, ineq line_theraml num viol 33.8431, eq max 0.0005, eq mean 0.0000


Epoch 28: train loss 6365.6602, train obj 722.7114, ineq max 197.1271, ineq mean 0.0561, ineq p_g num viol 0.9207, ineq q_g num viol 37.4948, ineq v_m num viol 0.0000, ineq line_theraml num viol 33.8431, eq max 0.0005, eq mean 0.0000
or small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are designed for small sizes. It might be better to use the
   Native/Hybrid classical routines if you want good performance.
   WARNING batched routines are design

Epoch 29: train loss 6169.1211, train obj 713.9714, ineq max 190.6342, ineq mean 0.0543, ineq p_g num viol 0.8950, ineq q_g num viol 36.5250, ineq v_m num viol 0.0000, ineq line_theraml num viol 32.7383, eq max 0.0005, eq mean 0.0000


Epoch 29: train loss 6169.1211, train obj 713.9714, ineq max 190.6342, ineq mean 0.0543, ineq p_g num viol 0.8950, ineq q_g num viol 36.5250, ineq v_m num viol 0.0000, ineq line_theraml num viol 32.7383, eq max 0.0005, eq mean 0.0000


In [19]:
# Save the training history
# train_loss_list
# epoch_stats['train_loss']
(np.array(train_loss_list)).tolist()
global_logger.info("train_loss_list:{}".format((np.array(train_loss_list)).tolist()))

data_tracking = {
                "train_loss_list": (np.array(train_loss_list)).tolist(),
                }
with open(os.path.join(data_tracking_path, "metrics.pickle"), 'wb') as handle:
    pickle.dump(data_tracking, handle, protocol=pickle.HIGHEST_PROTOCOL)

train_loss_list:[95748.9453125, 66563.5, 46373.75390625, 35335.11328125, 28804.134765625, 24440.787109375, 21272.41796875, 18807.16015625, 16852.26171875, 15271.1240234375, 14602.36328125, 13910.2490234375, 13097.96875, 12251.4912109375, 11545.169921875, 10871.0869140625, 10282.76953125, 9748.431640625, 9264.271484375, 8829.5732421875, 8500.15625, 8187.044921875, 7875.10888671875, 7575.93310546875, 7300.7978515625, 7041.03564453125, 6800.8916015625, 6575.2431640625, 6365.66015625, 6169.12109375]


* Evaluation (using validation set)

In [20]:
len(data.test_dataset)

200

In [12]:
# load pretrained_model
# solver_net = torch.load(r'./models/pretrained_models/graphlde_3970_pretrained_model_20_real_slack_chebconv.pt', weights_only=False)

# solver_net = torch.load(r'./models/pretrained_models/graphlde_3970_pretrained_model_20_real_slack_chebconv_update.pt', weights_only=False)
# solver_net = torch.load(r'./models/pretrained_models/graphlde_3970_pretrained_model_20_real_slack_chebconv_update_(weight_init_default).pt', weights_only=False)

# solver_net = torch.load(r'./models/pretrained_models/graphlde_3970_pretrained_model_20_real_slack_chebconv_test.pt', weights_only=False) # 경남형이 만들어준 데이터셋으로 학습한 모델 ==> 검증 필요....

In [21]:
from pypower.api import makeYbus
Ybus, Yf, Yt = makeYbus(data.baseMVA, data.ppc['bus'], data.ppc['branch'])
# branch thermal limit information
flow_max = (data.ppc['branch'][:, 5] / data.baseMVA)**2
flow_max[flow_max == 0] = np.inf # np.Inf
flow_max = torch.tensor(flow_max, dtype=torch.float32).to(data.device)

test_len = 0
node_means, node_stds, edge_means, edge_stds = data.input_standardization(test_len, train=False) # (1, 2*nbus) <= for data normalization
n_means = node_means.to(DEVICE)
n_stds = node_stds.to(DEVICE)
e_means = edge_means.to(DEVICE)
e_stds = edge_stds.to(DEVICE)

test_loader = torch_geometric.loader.DataLoader(data.test_dataset[test_len:], batch_size=1, shuffle=False, drop_last=True)
# test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)

solver_net.eval()
test_stats = {}
test_eps_converge = 1e-4

# LagM = torch.ones(1, 2*ng + 2*nbus + 2*nl).to(DEVICE) # shape: (1, num_inequalities)
LagM_sp_g = torch.ones(1, 2).to(DEVICE) # shape: (1, num_inequalities)
LagM_q_g = torch.ones(1, 2*ng).to(DEVICE) # shape: (1, num_inequalities)
LagM_v_m = torch.ones(1, 2*nbus).to(DEVICE) # shape: (1, num_inequalities)
LagM_line_l = torch.ones(1, 2*nl).to(DEVICE) # shape: (1, num_inequalities)

solve_time = []
for (i, Xtest) in enumerate(test_loader):
    Xtest = Xtest.to(DEVICE)

    start_time = time.time()
    Y = solver_net(Xtest, n_means, n_stds, e_means, e_stds)
    end_time = time.time()

    solve_time += [end_time - start_time]

    ## line thermal limit
    pg, qg, vm, va = data.get_yvars(Y)
    vr = vm*torch.cos(va)
    vi = vm*torch.sin(va)
    vz = torch.complex(vr, vi) # complex voltage

    # calculate the branch current of from bus and to bus based on the Yf*V and Yt*V
    If = torch.tensor(Yf.todense(), dtype=torch.complex64).to(data.device) @ vz.T
    It = torch.tensor(Yt.todense(), dtype=torch.complex64).to(data.device) @ vz.T

    # Calculate the apparent power S
    Sf = vz[:,data.ppc['branch'][:,0].astype(int)] * torch.conj(If.T)
    St = vz[:,data.ppc['branch'][:,1].astype(int)] * torch.conj(It.T)
    Sff = Sf * torch.conj(Sf)
    Stt = St * torch.conj(St)

    # calculate the line thermal limit constraints violation
    diff_Sf = Sff.real - flow_max
    diff_St = Stt.real - flow_max
    # diff_Sf[torch.clamp(diff_Sf, 0) != 0]

    line_limit_vio_Sf = torch.clamp(diff_Sf, 0)
    line_limit_vio_St = torch.clamp(diff_St, 0)
    ###########################################

    # test_loss, test_obj_cost, test_ineq_dist, test_eq_resid = total_loss(data, Xtest.x, Y, LagM)
    test_loss, test_obj_cost, test_ineq_dist, test_eq_resid = total_loss(data, Xtest.x, Y, LagM_sp_g, LagM_q_g, LagM_v_m, LagM_line_l) # LagM is lagrangian multiplier, and the shape is (1, num_inequalities)

    dict_agg(test_stats, 'time', end_time - start_time, op='sum')

    test_ineq_p_g = torch.cat([pg - data.pmax, data.pmin - pg], dim=1)
    test_ineq_p_g = torch.clamp(test_ineq_p_g, 0).to(data.device)
    test_ineq_q_g = test_ineq_dist[:,2:2+2*ng]
    test_ineq_v_m = test_ineq_dist[:,2+2*ng:2+2*ng+2*nbus]
    test_ineq_line_l = test_ineq_dist[:,2+2*ng+2*nbus:]

    dict_agg(test_stats, 'test_loss', test_loss.detach().cpu().numpy())
    # dict_agg(test_stats, 'test_loss', (test_loss[0]+test_loss[1]+test_loss[2]+test_loss[3]).detach().cpu().numpy())

    dict_agg(test_stats, 'test_obj_cost', test_obj_cost.detach().cpu().numpy())

    dict_agg(test_stats, 'test_ineq_max', torch.max(test_ineq_dist, dim=1)[0].detach().cpu().numpy())
    dict_agg(test_stats, 'test_ineq_mean', torch.mean(test_ineq_dist, dim=1).detach().cpu().numpy())
    dict_agg(test_stats, 'test_ineq_p_g_max', torch.max(test_ineq_p_g, dim=1)[0].detach().cpu().numpy())
    dict_agg(test_stats, 'test_ineq_p_g_mean', torch.mean(test_ineq_p_g, dim=1).detach().cpu().numpy())
    dict_agg(test_stats, 'test_ineq_q_g_max', torch.max(test_ineq_q_g, dim=1)[0].detach().cpu().numpy())
    dict_agg(test_stats, 'test_ineq_q_g_mean', torch.mean(test_ineq_q_g, dim=1).detach().cpu().numpy())
    dict_agg(test_stats, 'test_ineq_v_m_max', torch.max(test_ineq_v_m, dim=1)[0].detach().cpu().numpy())
    dict_agg(test_stats, 'test_ineq_v_m_mean', torch.mean(test_ineq_v_m, dim=1).detach().cpu().numpy())
    dict_agg(test_stats, 'test_ineq_line_l_max', torch.max(test_ineq_line_l, dim=1)[0].detach().cpu().numpy())
    dict_agg(test_stats, 'test_ineq_line_l_mean', torch.mean(test_ineq_line_l, dim=1).detach().cpu().numpy())

    pg_rate_torch = (((pg <= data.pmax) & (pg >= data.pmin)).sum()/ng)*100
    qg_rate_torch = (((qg <= data.qmax) & (qg >= data.qmin)).sum()/ng)*100
    dict_agg(test_stats, 'test_p_g_satisfication rate (%)', pg_rate_torch.detach().cpu().numpy().reshape(-1,1))
    dict_agg(test_stats, 'test_q_g_satisfication rate (%)', qg_rate_torch.detach().cpu().numpy().reshape(-1,1))
    # dict_agg(test_stats, 'test_p_g_satisfication rate (%)', ((torch.sum(test_ineq_p_g == 0, dim=1)/test_ineq_p_g.shape[1])*100).detach().cpu().numpy())
    # dict_agg(test_stats, 'test_q_g_satisfication rate (%)', ((torch.sum(test_ineq_q_g == 0, dim=1)/test_ineq_q_g.shape[1])*100).detach().cpu().numpy())

    v_rate_torch = (((vm <= data.vmax) & (vm >= data.vmin)).sum()/nbus)*100
    dict_agg(test_stats, 'test_v_m_satisfication rate (%)', v_rate_torch.detach().cpu().numpy().reshape(-1,1))
    # dict_agg(test_stats, 'test_v_m_satisfication rate (%)', ((torch.sum(test_ineq_v_m == 0, dim=1)/test_ineq_v_m.shape[1])*100).detach().cpu().numpy())

    sff_rate_torch = ((Sff.real <= flow_max).sum()/nl)*100        
    stt_rate_torch = ((Stt.real <= flow_max).sum()/nl)*100        
    dict_agg(test_stats, 'test_line_limit_satisfication_rate_Sf(%)', sff_rate_torch.detach().cpu().numpy().reshape(-1,1))
    dict_agg(test_stats, 'test_line_limit_satisfication_rate_St(%)', stt_rate_torch.detach().cpu().numpy().reshape(-1,1))
    dict_agg(test_stats, 'test_line_limit_satisfication_rate (%)', ((torch.sum(test_ineq_line_l == 0, dim=1)/test_ineq_line_l.shape[1])*100).detach().cpu().numpy())
    # dict_agg(test_stats, 'test_line_limit_satisfication_rate_Sf(%)', ((torch.sum(line_limit_vio_Sf == 0, dim=1)/line_limit_vio_Sf.shape[1])*100).detach().cpu().numpy())
    # dict_agg(test_stats, 'test_line_limit_satisfication_rate_St(%)', ((torch.sum(line_limit_vio_St == 0, dim=1)/line_limit_vio_St.shape[1])*100).detach().cpu().numpy())

    dict_agg(test_stats, 'test_ineq_q_g_num_viol_0', torch.sum(test_ineq_q_g > test_eps_converge, dim=1).detach().cpu().numpy())
    dict_agg(test_stats, 'test_ineq_v_m_num_viol_0', torch.sum(test_ineq_v_m > test_eps_converge, dim=1).detach().cpu().numpy())

    test_eq_real = test_eq_resid[:,:nbus]
    test_eq_react = test_eq_resid[:,nbus:]
    dict_agg(test_stats, 'test_eq_max', torch.max(torch.abs(test_eq_resid), dim=1)[0].detach().cpu().numpy())
    dict_agg(test_stats, 'test_eq_mean', torch.mean(torch.abs(test_eq_resid), dim=1).detach().cpu().numpy())
    dict_agg(test_stats, 'test_eq_real_max', torch.max(torch.abs(test_eq_real), dim=1)[0].detach().cpu().numpy())
    dict_agg(test_stats, 'test_eq_real_mean', torch.mean(torch.abs(test_eq_real), dim=1).detach().cpu().numpy())
    dict_agg(test_stats, 'test_eq_react_max', torch.max(torch.abs(test_eq_react), dim=1)[0].detach().cpu().numpy())
    dict_agg(test_stats, 'test_eq_react_mean', torch.mean(torch.abs(test_eq_react), dim=1).detach().cpu().numpy())
    dict_agg(test_stats, 'test_active_eq_satisfication rate (%)', (torch.sum((test_eq_resid[:,:nbus] <= 1e-2) & (test_eq_resid[:,:nbus] >= -1e-2)  , dim=1)/test_eq_resid[:,:nbus].shape[1]*100).detach().cpu().numpy())
    dict_agg(test_stats, 'test_reactive_eq_satisfication rate (%)', (torch.sum((test_eq_resid[:,nbus:] <= 1e-2) & (test_eq_resid[:,nbus:] >= -1e-2)  , dim=1)/test_eq_resid[:,nbus:].shape[1]*100).detach().cpu().numpy())

    print('Test batch {}: test loss {:.4f}, test obj {:.4f}, ineq max {:.4f}, ineq mean {:.4f}, ineq q_g num viol {:.4f}, ineq v_m num viol {:.4f}, eq max {:.4f}, eq mean {:.4f}, p_g satisfication rate {:.4f}, q_g satisfication rate {:.4f}, v_m satisfication rate {:.4f}, test_line_limit_satisfication_rate {:.4f}, test_line_limit_satisfication_rate_Sf {:.4f}, test_line_limit_satisfication_rate_St {:.4f}, active eq satisfication rate {:.4f}, reactive eq satisfication rate {:.4f}'.format(
                i, np.mean(test_stats['test_loss']), np.mean(test_stats['test_obj_cost']), np.mean(test_stats['test_ineq_max']), np.mean(test_stats['test_ineq_mean']),
                np.mean(test_stats['test_ineq_q_g_num_viol_0']), np.mean(test_stats['test_ineq_v_m_num_viol_0']),
                np.mean(test_stats['test_eq_max']), np.mean(test_stats['test_eq_mean']), np.mean(test_stats['test_p_g_satisfication rate (%)']), np.mean(test_stats['test_q_g_satisfication rate (%)']), np.mean(test_stats['test_v_m_satisfication rate (%)']), np.mean(test_stats['test_line_limit_satisfication_rate (%)']), np.mean(test_stats['test_line_limit_satisfication_rate_Sf(%)']), np.mean(test_stats['test_line_limit_satisfication_rate_St(%)']), np.mean(test_stats['test_active_eq_satisfication rate (%)']), np.mean(test_stats['test_reactive_eq_satisfication rate (%)'])))

#     print('Test batch {}: test loss {:.4f}, ineq max {:.4f}, ineq mean {:.4f}, ineq q_g num viol {:.4f}, ineq v_m num viol {:.4f}, line_limit satisfication rate Sf {:.4f}, line_limit satisfication rate St {:.4f}, eq max {:.4f}, eq mean {:.4f}'.format(
#             i, np.mean(test_stats['test_loss']), np.mean(test_stats['test_ineq_max']), np.mean(test_stats['test_ineq_mean']),
#             np.mean(test_stats['test_ineq_q_g_num_viol_0']), np.mean(test_stats['test_ineq_v_m_num_viol_0']), np.mean(test_stats['test_line_limit_satisfication_rate_Sf(%)']),
#             np.mean(test_stats['test_line_limit_satisfication_rate_St(%)']), np.mean(test_stats['test_eq_max']), np.mean(test_stats['test_eq_mean'])))


Test batch 0: test loss 469.2130, test obj 456.8150, ineq max 2.5580, ineq mean 0.0006, ineq q_g num viol 8.0000, ineq v_m num viol 0.0000, eq max 0.0006, eq mean 0.0000, p_g satisfication rate 100.0000, q_g satisfication rate 93.4959, v_m satisfication rate 100.0000, test_line_limit_satisfication_rate 99.9849, test_line_limit_satisfication_rate_Sf 99.9849, test_line_limit_satisfication_rate_St 99.9849, active eq satisfication rate 100.0000, reactive eq satisfication rate 100.0000
Test batch 1: test loss 469.5019, test obj 457.0653, ineq max 2.5664, ineq mean 0.0006, ineq q_g num viol 8.5000, ineq v_m num viol 0.0000, eq max 0.0005, eq mean 0.0000, p_g satisfication rate 100.0000, q_g satisfication rate 93.0894, v_m satisfication rate 100.0000, test_line_limit_satisfication_rate 99.9849, test_line_limit_satisfication_rate_Sf 99.9849, test_line_limit_satisfication_rate_St 99.9849, active eq satisfication rate 100.0000, reactive eq satisfication rate 100.0000
Test batch 2: test loss 469.

* Arithmetic mean

In [22]:
## Calculate the results of GraphLDE

print("GraphLDE obj. value for test samples: ", round(np.mean(test_stats['test_obj_cost'])*10000, 4))
print("GraphLDE eq. mean for test samples: ", np.mean(test_stats['test_eq_mean']))
print("GraphLDE eq. max for test samples: ", np.mean(test_stats['test_eq_max']))
print("GraphLDE eq. active mean for test samples: ", np.mean(test_stats['test_eq_real_mean']))
print("GraphLDE eq. active max for test samples: ", np.mean(test_stats['test_eq_real_max']))
print("GraphLDE eq. reactive mean for test samples: ", np.mean(test_stats['test_eq_react_mean']))
print("GraphLDE eq. reactive max for test samples: ", np.mean(test_stats['test_eq_react_max']))

print("\n")
print("GraphLDE ineq. mean for test samples: ", np.mean(test_stats['test_ineq_mean']))
print("GraphLDE ineq. max for test samples: ", np.mean(test_stats['test_ineq_max']))
print("GraphLDE ineq. p_g mean for test samples: ", np.mean(test_stats['test_ineq_p_g_mean']))
print("GraphLDE ineq. p_g max for test samples: ", np.mean(test_stats['test_ineq_p_g_max']))
print("GraphLDE ineq. q_g mean for test samples: ", np.mean(test_stats['test_ineq_q_g_mean']))
print("GraphLDE ineq. q_g max for test samples: ", np.mean(test_stats['test_ineq_q_g_max']))
print("GraphLDE ineq. v_m mean for test samples: ", np.mean(test_stats['test_ineq_v_m_mean']))
print("GraphLDE ineq. v_m max for test samples: ", np.mean(test_stats['test_ineq_v_m_max']))
print("GraphLDE ineq. line_l mean for test samples: ", np.mean(test_stats['test_ineq_line_l_mean']))
print("GraphLDE ineq. line_l max for test samples: ", np.mean(test_stats['test_ineq_line_l_max']))

print("\n")
print("GraphLDE p_g satisfication rate for test samples: ", np.mean(test_stats['test_p_g_satisfication rate (%)']))
print("GraphLDE q_g satisfication rate for test samples: ", np.mean(test_stats['test_q_g_satisfication rate (%)']))
print("GraphLDE v_m satisfication rate for test samples: ", np.mean(test_stats['test_v_m_satisfication rate (%)']))
print("GraphLDE test_line_limit_satisfication_rate_Sf for test samples: ", np.mean(test_stats['test_line_limit_satisfication_rate_Sf(%)']))
print("GraphLDE test_line_limit_satisfication_rate_St for test samples: ", np.mean(test_stats['test_line_limit_satisfication_rate_St(%)']))
print("GraphLDE active eq satisfication rate for test samples: ", np.mean(test_stats['test_active_eq_satisfication rate (%)']))
print("GraphLDE reactive eq satisfication rate for test samples: ", np.mean(test_stats['test_reactive_eq_satisfication rate (%)']))

print("\n")
# print("DeepLDE time (ms) <== average value for test dataset:", (test_stats['time']/1000)*1e3)
print("GraphLDE time (ms) <== average value for test dataset:", (np.mean(solve_time)/1)*1e3)

global_logger.info('GraphLDE obj. value for test samples: {}'.format(round(np.mean(test_stats['test_obj_cost'])*10000, 4)))
global_logger.info('GraphLDE eq. mean for test samples: {}'.format(np.mean(test_stats['test_eq_mean'])))
global_logger.info('GraphLDE eq. max for test samples: {}'.format(np.mean(test_stats['test_eq_max'])))
global_logger.info('GraphLDE eq. active mean for test samples: {}'.format(np.mean(test_stats['test_eq_real_mean'])))
global_logger.info('GraphLDE eq. active max for test samples: {}'.format(np.mean(test_stats['test_eq_real_max'])))
global_logger.info('GraphLDE eq. reactive mean for test samples: {}'.format(np.mean(test_stats['test_eq_react_mean'])))
global_logger.info('GraphLDE eq. reactive max for test samples: {}'.format(np.mean(test_stats['test_eq_react_max'])))
global_logger.info('\nGraphLDE ineq. mean for test samples: {}'.format(np.mean(test_stats['test_ineq_mean'])))
global_logger.info('GraphLDE ineq. max for test samples: {}'.format(np.mean(test_stats['test_ineq_max'])))
global_logger.info('GraphLDE ineq. p_g mean for test samples: {}'.format(np.mean(test_stats['test_ineq_p_g_mean'])))
global_logger.info('GraphLDE ineq. p_g max for test samples: {}'.format(np.mean(test_stats['test_ineq_p_g_max'])))
global_logger.info('GraphLDE ineq. q_g mean for test samples: {}'.format(np.mean(test_stats['test_ineq_q_g_mean'])))
global_logger.info('GraphLDE ineq. q_g max for test samples: {}'.format(np.mean(test_stats['test_ineq_q_g_max'])))
global_logger.info('GraphLDE ineq. v_m mean for test samples: {}'.format(np.mean(test_stats['test_ineq_v_m_mean'])))
global_logger.info('GraphLDE ineq. v_m max for test samples: {}'.format(np.mean(test_stats['test_ineq_v_m_max'])))
global_logger.info('GraphLDE ineq. line_l mean for test samples: {}'.format(np.mean(test_stats['test_ineq_line_l_mean'])))
global_logger.info('GraphLDE ineq. line_l max for test samples: {}'.format(np.mean(test_stats['test_ineq_line_l_max'])))
global_logger.info('\nGraphLDE p_g satisfication rate for test samples: {}'.format(np.mean(test_stats['test_p_g_satisfication rate (%)'])))
global_logger.info('GraphLDE q_g satisfication rate for test samples: {}'.format(np.mean(test_stats['test_q_g_satisfication rate (%)'])))
global_logger.info('GraphLDE v_m satisfication rate for test samples: {}'.format(np.mean(test_stats['test_v_m_satisfication rate (%)'])))
global_logger.info('GraphLDE test_line_limit_satisfication_rate_Sf for test samples: {}'.format(np.mean(test_stats['test_line_limit_satisfication_rate_Sf(%)'])))
global_logger.info('GraphLDE test_line_limit_satisfication_rate_St for test samples: {}'.format(np.mean(test_stats['test_line_limit_satisfication_rate_St(%)'])))
global_logger.info('GraphLDE active eq satisfication rate for test samples: {}'.format(np.mean(test_stats['test_active_eq_satisfication rate (%)'])))
global_logger.info('GraphLDE reactive eq satisfication rate for test samples: {}'.format(np.mean(test_stats['test_reactive_eq_satisfication rate (%)'])))
global_logger.info('\nGraphLDE time (ms) <== average value for test dataset: {}'.format((np.mean(solve_time)/1)*1e3))


GraphLDE obj. value for test samples: 4586152.5
GraphLDE eq. mean for test samples: 7.57376074034255e-06
GraphLDE eq. max for test samples: 0.0005095619126223028
GraphLDE eq. active mean for test samples: 3.495321834634524e-06
GraphLDE eq. active max for test samples: 0.0001369700621580705
GraphLDE eq. reactive mean for test samples: 1.1652200555545278e-05
GraphLDE eq. reactive max for test samples: 0.0005095619126223028

GraphLDE ineq. mean for test samples: 0.0005791621515527368
GraphLDE ineq. max for test samples: 2.5440661907196045
GraphLDE ineq. p_g mean for test samples: 0.00015819602413102984
GraphLDE ineq. p_g max for test samples: 0.03891622647643089
GraphLDE ineq. q_g mean for test samples: 0.02973296120762825
GraphLDE ineq. q_g max for test samples: 2.272455930709839
GraphLDE ineq. v_m mean for test samples: 0.0
GraphLDE ineq. v_m max for test samples: 0.0
GraphLDE ineq. line_l mean for test samples: 0.00038257683627307415
GraphLDE ineq. line_l max for test samples: 2.544066

GraphLDE obj. value for test samples:  4586152.5
GraphLDE eq. mean for test samples:  7.5737607e-06
GraphLDE eq. max for test samples:  0.0005095619
GraphLDE eq. active mean for test samples:  3.4953218e-06
GraphLDE eq. active max for test samples:  0.00013697006
GraphLDE eq. reactive mean for test samples:  1.1652201e-05
GraphLDE eq. reactive max for test samples:  0.0005095619


GraphLDE ineq. mean for test samples:  0.00057916215
GraphLDE ineq. max for test samples:  2.5440662
GraphLDE ineq. p_g mean for test samples:  0.00015819602
GraphLDE ineq. p_g max for test samples:  0.038916226
GraphLDE ineq. q_g mean for test samples:  0.029732961
GraphLDE ineq. q_g max for test samples:  2.272456
GraphLDE ineq. v_m mean for test samples:  0.0
GraphLDE ineq. v_m max for test samples:  0.0
GraphLDE ineq. line_l mean for test samples:  0.00038257684
GraphLDE ineq. line_l max for test samples:  2.5440662


GraphLDE p_g satisfication rate for test samples:  99.95121
GraphLDE q_g satisfication r

* Harmonic mean

In [ ]:
import statistics as st

## Calculate optimality gap
print("GraphLDE obj. value for test samples: ", round(np.mean(test_stats['test_obj_cost'])*10000, 4))
print("GraphLDE eq. mean for test samples: ", st.harmonic_mean(test_stats['test_eq_mean'])) # print("LDF eq. mean for test samples: ", np.mean(test_stats['test_eq_mean']))
print("GraphLDE eq. max for test samples: ", st.harmonic_mean(test_stats['test_eq_max'])) # print("LDF eq. max for test samples: ", np.mean(test_stats['test_eq_max']))
print("GraphLDE eq. active mean for test samples: ", st.harmonic_mean(test_stats['test_eq_real_mean'])) # print("LDF eq. active mean for test samples: ", np.mean(test_stats['test_eq_real_mean']))
print("GraphLDE eq. active max for test samples: ", st.harmonic_mean(test_stats['test_eq_real_max'])) # print("LDF eq. active max for test samples: ", np.mean(test_stats['test_eq_real_max']))
print("GraphLDE eq. reactive mean for test samples: ", st.harmonic_mean(test_stats['test_eq_react_mean'])) # print("LDF eq. reactive mean for test samples: ", np.mean(test_stats['test_eq_react_mean']))
print("GraphLDE eq. reactive max for test samples: ", st.harmonic_mean(test_stats['test_eq_react_max'])) # print("LDF eq. reactive max for test samples: ", np.mean(test_stats['test_eq_react_max']))
print("\n")
print("GraphLDE ineq. mean for test samples: ", st.harmonic_mean(test_stats['test_ineq_mean'])) # print("LDF ineq. mean for test samples: ", np.mean(test_stats['test_ineq_mean']))
print("GraphLDE ineq. max for test samples: ", st.harmonic_mean(test_stats['test_ineq_max'])) # print("LDF ineq. max for test samples: ", np.mean(test_stats['test_ineq_max']))
print("GraphLDE ineq. p_g mean for test samples: ", st.harmonic_mean(test_stats['test_ineq_p_g_mean'])) # print("LDF ineq. p_g mean for test samples: ", np.mean(test_stats['test_ineq_p_g_mean']))
print("GraphLDE ineq. p_g max for test samples: ", st.harmonic_mean(test_stats['test_ineq_p_g_max'])) # print("LDF ineq. p_g max for test samples: ", np.mean(test_stats['test_ineq_p_g_max']))
print("GraphLDE ineq. q_g mean for test samples: ", st.harmonic_mean(test_stats['test_ineq_q_g_mean'])) # print("LDF ineq. q_g mean for test samples: ", np.mean(test_stats['test_ineq_q_g_mean']))
print("GraphLDE ineq. q_g max for test samples: ", st.harmonic_mean(test_stats['test_ineq_q_g_max'])) # print("LDF ineq. q_g max for test samples: ", np.mean(test_stats['test_ineq_q_g_max']))
print("GraphLDE ineq. v_m mean for test samples: ", st.harmonic_mean(test_stats['test_ineq_v_m_mean'])) # print("LDF ineq. v_m mean for test samples: ", np.mean(test_stats['test_ineq_v_m_mean']))
print("GraphLDE ineq. v_m max for test samples: ", st.harmonic_mean(test_stats['test_ineq_v_m_max'])) # print("LDF ineq. v_m max for test samples: ", np.mean(test_stats['test_ineq_v_m_max']))
print("GraphLDE ineq. line_l mean for test samples: ", st.harmonic_mean(test_stats['test_ineq_line_l_mean'])) # print("LDF ineq. line_l mean for test samples: ", np.mean(test_stats['test_ineq_line_l_mean']))
print("GraphLDE ineq. line_l max for test samples: ", st.harmonic_mean(test_stats['test_ineq_line_l_max'])) # print("LDF ineq. line_l max for test samples: ", np.mean(test_stats['test_ineq_line_l_max']))
print("\n")

print("GraphLDE p_g satisfication rate for test samples: ", st.harmonic_mean(test_stats['test_p_g_satisfication rate (%)'].reshape(-1))) # print("LDF p_g satisfication rate for test samples: ", np.mean(test_stats['test_p_g_satisfication rate (%)']))
print("GraphLDE q_g satisfication rate for test samples: ", st.harmonic_mean(test_stats['test_q_g_satisfication rate (%)'].reshape(-1))) # print("LDF q_g satisfication rate for test samples: ", np.mean(test_stats['test_q_g_satisfication rate (%)']))
print("GraphLDE v_m satisfication rate for test samples: ", st.harmonic_mean(test_stats['test_v_m_satisfication rate (%)'].reshape(-1))) # print("LDF v_m satisfication rate for test samples: ", np.mean(test_stats['test_v_m_satisfication rate (%)']))
print("GraphLDE test_line_limit_satisfication_rate_Sf for test samples: ", st.harmonic_mean(test_stats['test_line_limit_satisfication_rate_Sf(%)'].reshape(-1))) # print("LDF test_line_limit_satisfication_rate_Sf for test samples: ", np.mean(test_stats['test_line_limit_satisfication_rate_Sf(%)']))
print("GraphLDE test_line_limit_satisfication_rate_St for test samples: ", st.harmonic_mean(test_stats['test_line_limit_satisfication_rate_St(%)'].reshape(-1))) # print("LDF test_line_limit_satisfication_rate_St for test samples: ", np.mean(test_stats['test_line_limit_satisfication_rate_St(%)']))
print("GraphLDE active eq satisfication rate for test samples: ", st.harmonic_mean(test_stats['test_active_eq_satisfication rate (%)'].reshape(-1))) # print("LDF active eq satisfication rate for test samples: ", np.mean(test_stats['test_active_eq_satisfication rate (%)']))
print("GraphLDE reactive eq satisfication rate for test samples: ", st.harmonic_mean(test_stats['test_reactive_eq_satisfication rate (%)'].reshape(-1))) # print("LDF reactive eq satisfication rate for test samples: ", np.mean(test_stats['test_reactive_eq_satisfication rate (%)']))

print("\n")
print("GraphLDE time (ms) <== average value for test dataset:", (np.mean(solve_time)/1)*1e3)


In [ ]:
# optimality gap
################### UNIFORM DISTRIBUTION - RANDOM SAMPLING ###################
## +/-20% perturbation <== Training time: 38min (min)
# MATPOWER: cost - 3830500  // solve time - 5341.10 (ms)
# GraphLDE: cost - 3862003.1738 // solve time - 168.9411700963974 (ms)

((3862003.1738-3830500)/3830500)*100